# Graph Connectivity: SFT + Auxiliary DSU Loss

Эксперименты для диплома. Воспроизведение идеи auxiliary loss (Bai et al.) на задаче graph reachability с DSU state-tracking.

**Стадии:**
1. Генерация данных (графы всех типов)
2. SFT Baseline (только classification loss)
3. Auxiliary Loss (classification + DSU state-tracking)
4. Evaluation (ID + OOD)

**Запуск:** Kaggle GPU (T4/P100)

## 0. Setup

In [ ]:
!git clone https://github.com/maximvw/transformer-analyzing.git /content/diplom
%cd /content/diplom

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Генерация данных

20k train графов (on-the-fly augmentation), 2k val, 2k test ID, 1k на каждый OOD тест.

In [ ]:
!python graph_connectivity/scripts/generate_data.py \
    --output_dir graph_connectivity/data \
    --n_train 20000 \
    --n_val 2000 \
    --n_test 2000 \
    --seed 42

In [ ]:
# Проверка: сколько файлов сгенерировалось
import os, json
data_dir = "graph_connectivity/data"
for f in sorted(os.listdir(data_dir)):
    path = os.path.join(data_dir, f)
    with open(path) as fh:
        data = json.load(fh)
    print(f"{f}: {len(data)} examples")

## 2. Стадия 1: SFT Baseline

4L4H GPT-2 (d_model=256), обучение с нуля. Только classification loss на `<ANS>` позиции. `lambda_state=0` (без auxiliary loss).

In [ ]:
!python -m graph_connectivity.src.train \
    --train_path graph_connectivity/data/train.json \
    --val_path graph_connectivity/data/val.json \
    --max_n 30 \
    --d_model 256 --n_layer 4 --n_head 4 \
    --lr 5e-5 --weight_decay 0.01 \
    --batch_size 32 --epochs 50 --patience 10 \
    --lambda_state 0.0 \
    --save_dir checkpoints/graph_sft \
    --seed 42

## 3. Стадия 2: Auxiliary Loss (DSU state-tracking)

Тот же GPT-2 4L4H, но с DSU probe. `lambda_state` — вес auxiliary loss. Начинаем с `lambda=1.0`.

In [ ]:
!python -m graph_connectivity.src.train \
    --train_path graph_connectivity/data/train.json \
    --val_path graph_connectivity/data/val.json \
    --max_n 30 \
    --d_model 256 --n_layer 4 --n_head 4 \
    --lr 5e-5 --weight_decay 0.01 \
    --batch_size 32 --epochs 50 --patience 10 \
    --lambda_state 1.0 \
    --save_dir checkpoints/graph_aux_lambda1.0 \
    --seed 42

### 3b. Grid search по lambda (опционально)

Раскомментируй нужные значения lambda.

In [ ]:
# for lam in [0.1, 0.5, 2.0, 5.0]:
#     !python -m graph_connectivity.src.train \
#         --train_path graph_connectivity/data/train.json \
#         --val_path graph_connectivity/data/val.json \
#         --max_n 30 \
#         --d_model 256 --n_layer 4 --n_head 4 \
#         --lr 5e-5 --weight_decay 0.01 \
#         --batch_size 32 --epochs 50 --patience 10 \
#         --lambda_state {lam} \
#         --save_dir checkpoints/graph_aux_lambda{lam} \
#         --seed 42

## 4. Evaluation (ID + OOD)

In [ ]:
from graph_connectivity.src.evaluate import evaluate_all

print("=" * 60)
print("SFT Baseline")
print("=" * 60)
sft_results = evaluate_all(
    checkpoint_path="checkpoints/graph_sft/checkpoint_best.pt",
    data_dir="graph_connectivity/data",
    lambda_state=0.0,
)

print()
print("=" * 60)
print("Auxiliary Loss (lambda=1.0)")
print("=" * 60)
aux_results = evaluate_all(
    checkpoint_path="checkpoints/graph_aux_lambda1.0/checkpoint_best.pt",
    data_dir="graph_connectivity/data",
    lambda_state=1.0,
)

## 5. Визуализация learning curves

In [ ]:
import json
import matplotlib.pyplot as plt

def plot_training_log(log_path, label):
    with open(log_path) as f:
        log = json.load(f)
    epochs = [e["epoch"] for e in log]
    train_acc = [e["train"]["accuracy"] for e in log]
    val_acc = [e["val"]["accuracy"] for e in log]
    train_loss = [e["train"]["loss"] for e in log]
    val_loss = [e["val"]["loss"] for e in log]
    return epochs, train_acc, val_acc, train_loss, val_loss

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SFT
try:
    ep, ta, va, tl, vl = plot_training_log("checkpoints/graph_sft/train_log.json", "SFT")
    axes[0].plot(ep, ta, label="SFT train", linestyle="--", color="C0")
    axes[0].plot(ep, va, label="SFT val", color="C0")
    axes[1].plot(ep, tl, label="SFT train", linestyle="--", color="C0")
    axes[1].plot(ep, vl, label="SFT val", color="C0")
except FileNotFoundError:
    print("SFT log not found")

# Aux Loss
try:
    ep, ta, va, tl, vl = plot_training_log("checkpoints/graph_aux_lambda1.0/train_log.json", "Aux")
    axes[0].plot(ep, ta, label="Aux train", linestyle="--", color="C1")
    axes[0].plot(ep, va, label="Aux val", color="C1")
    axes[1].plot(ep, tl, label="Aux train", linestyle="--", color="C1")
    axes[1].plot(ep, vl, label="Aux val", color="C1")
except FileNotFoundError:
    print("Aux log not found")

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 6. Сравнительная таблица результатов

In [ ]:
import pandas as pd

rows = []
for name, results in [("SFT", sft_results), ("Aux Loss (λ=1.0)", aux_results)]:
    row = {"Model": name}
    for test_name, metrics in results.items():
        row[test_name] = f"{metrics['accuracy']:.2%}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Model")
print(df.to_string())
df

## 7. Сохранение чекпоинтов (Google Drive)

In [ ]:
# Раскомментируй для сохранения на Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r checkpoints /content/drive/MyDrive/diplom_graph_checkpoints/
# !cp training_curves.png /content/drive/MyDrive/diplom_graph_checkpoints/